# Creating and Appending to the Feature File

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os

pth = '../'
sys.path.append(os.path.dirname(pth))

In [4]:
from mltdm.den_fx import fx_feat

In [5]:
feat = fx_feat.append_feat('2026-03-05')

Download and loading daily save files from 2026-02-25-2026-03-05


 22%|██▏       | 2/9 [00:00<00:02,  2.86it/s]

Could not download https://lasp.colorado.edu/eve/data_access/eve_data/fism/flare_bands/2026/FISM_bands_2026057_v02_01.sav
A HTTPError was thrown: 404 Not Found


 33%|███▎      | 3/9 [00:01<00:01,  3.10it/s]

Could not download https://lasp.colorado.edu/eve/data_access/eve_data/fism/flare_bands/2026/FISM_bands_2026058_v02_01.sav
A HTTPError was thrown: 404 Not Found


...

 89%|████████▉ | 8/9 [00:02<00:00,  3.43it/s]

Could not download https://lasp.colorado.edu/eve/data_access/eve_data/fism/flare_bands/2026/FISM_bands_2026063_v02_01.sav
A HTTPError was thrown: 404 Not Found


100%|██████████| 9/9 [00:02<00:00,  3.28it/s]


Could not download https://lasp.colorado.edu/eve/data_access/eve_data/fism/flare_bands/2026/FISM_bands_2026064_v02_01.sav
A HTTPError was thrown: 404 Not Found


100%|██████████| 106/106 [00:04<00:00, 21.92it/s]


In [6]:
feat

,DateTime,1300_02,43000_09,85550_13,94400_18,SYM_H index,AE
0,2003-01-01 00:00:00,4.450318e+07,1.205613e+10,3.513738e+09,2.023032e+09,-5.0,24.0
1,2003-01-01 00:05:00,4.470757e+07,1.151408e+10,3.436102e+09,2.025284e+09,-4.0,20.0
2,2003-01-01 00:10:00,4.470341e+07,1.151807e+10,3.436713e+09,2.025301e+09,-4.0,26.0
3,2003-01-01 00:15:00,4.469097e+07,1.151813e+10,3.435544e+09,2.024837e+09,-3.0,24.0
4,2003-01-01 00:20:00,4.467231e+07,1.151469e+10,3.432776e+09,2.023948e+09,-3.0,27.0
...,...,...,...,...,...,...,...
2388955,2026-02-25 23:35:00,5.065073e+07,8.715655e+09,2.874630e+09,1.725895e+09,-5.0,NaN
2388956,2026-02-25 23:40:00,5.061547e+07,8.711789e+09,2.873099e+09,1.725367e+09,-5.0,NaN
2388957,2026-02-25 23:45:00,5.061056e+07,8.710639e+09,2.872638e+09,1.725234e+09,-6.0,NaN
2388958,2026-02-25 23:50:00,5.060464e+07,8.709434e+09,2.872153e+09,1.725093e+09,-6.0,NaN


In [5]:
def stream_kyoto_dst(url: str):
    
    id = (0,3)
    year_yy = (3,5)
    month = (5,7)
    day = (8,10)
    year_xx = (14,16)
    dat = [(20,24),(24,28),(28,32),(32,36),(36,40),(40,44),(44,48),(48,52),
        (52,56),(56,60),(60,64),(64,68),(68,72),(72,76),(76,80),(80,84),(84,88),
        (88,92),(92,96),(96,100),(100,104),(104,108),(108,112),(112,116)]
    avg = (116,120)


    l_df = []

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if line:
                line = line.decode('utf-8')
                if line[0] == '[':
                    break
                id_val = line[slice(*id)]
                yr = line[slice(*year_xx)]+line[slice(*year_yy)]
                mm = line[slice(*month)]
                dd = line[slice(*day)]
                dst = [line[slice(*s)] for s in dat]

                tr = pd.date_range(start=f'{yr}-{mm}-{dd}', periods=24, freq='h')

                l_df.append(pd.DataFrame({'DateTime':tr,'DST':dst}))

    return pd.concat(l_df, ignore_index=True)

In [ ]:
import pandas as pd
import numpy as np

import requests

import datetime
from dateutil.relativedelta import relativedelta

from mltdm.io import omni

def load_prelim_feat(sdate: str=None, edate: str=None):

    # kyoto base URL for DST data
    kyoto_base = 'https://wdc.kugi.kyoto-u.ac.jp/dst_realtime/'
    # load omni data first
    # drop rows where Sym-H is NaN (this is the variable typically used)
    omni_cols = ['SYM_H index', 'AE', 'DateTime']
    omni_data = omni(sdate=sdate, edate=edate)[omni_cols].dropna(subset=['SYM_H index'])
    
    # check what data we are missing
    # if we are missing more than 1 hour
    # data fill it with data from Kyoto (which is more complete but less accurate)
    ky_df = -1
    print((pd.to_datetime(edate)-omni_data['DateTime'].max()).total_seconds())
    if (pd.to_datetime(edate)-omni_data['DateTime'].max()).total_seconds() > 3600.:
        dt_ran = pd.date_range(start=omni_data['DateTime'].max(),end=edate, freq='MS')    
        kyoto_urls = [f'{kyoto_base}/{dt.strftime("%Y%m")}/dst{dt.strftime("%y%m")}.for.request' for dt in dt_ran]
        df_l = [stream_kyoto_dst(url) for url in kyoto_urls]

        ky_df = pd.concat(df_l, ignore_index=True)

        ky_df['SYM_H index'] = ky_df['DST'].astype(float)
        ky_df['AE'] = np.nan

        gd_val = (ky_df['DST'] != '9999')

        gd_t = (ky_df['DateTime'] > omni_data['DateTime'].max()) & (ky_df['DateTime'] <= pd.to_datetime(edate))

        ky_df = ky_df[gd_t & gd_val]

        omni_data = pd.concat([omni_data, ky_df[['DateTime','SYM_H index','AE']]], ignore_index=True)



    return omni_data, ky_df

    today = datetime.date.today()
    last_month_date = today - relativedelta(months=1)
    last_month = last_month_date.month
    last_year = last_month_date.year

    print(today)
    print(last_month_date)
    #print(last_month)
    #print(last_year)

    kyoto_curr = 'https://wdc.kugi.kyoto-u.ac.jp/dst_realtime/presentmonth/dst2603.for.request'
    kyoto_last = 'https://wdc.kugi.kyoto-u.ac.jp/dst_realtime/202602/dst2602.for.request'



In [39]:
ky_df = load_prelim_feat(sdate='2026-01-01', edate='2026-03-06')

100%|██████████| 106/106 [00:05<00:00, 19.17it/s]


432300.0


In [40]:
ky_df

,DateTime,DST,SYM_H index,AE
0,2026-03-01 00:00:00,3,3.0,NaN
1,2026-03-01 01:00:00,0,0.0,NaN
2,2026-03-01 02:00:00,-5,-5.0,NaN
3,2026-03-01 03:00:00,-6,-6.0,NaN
4,2026-03-01 04:00:00,-2,-2.0,NaN
...,...,...,...,...
739,2026-03-31 19:00:00,9999,9999.0,NaN
740,2026-03-31 20:00:00,9999,9999.0,NaN
741,2026-03-31 21:00:00,9999,9999.0,NaN
742,2026-03-31 22:00:00,9999,9999.0,NaN


In [28]:
ky_df

,DateTime,DST,SYM_H index,AE
0,2026-03-01 00:00:00,3,3.0,NaN
1,2026-03-01 01:00:00,0,0.0,NaN
2,2026-03-01 02:00:00,-5,-5.0,NaN
3,2026-03-01 03:00:00,-6,-6.0,NaN
4,2026-03-01 04:00:00,-2,-2.0,NaN
...,...,...,...,...
116,2026-03-05 20:00:00,21,21.0,NaN
117,2026-03-05 21:00:00,17,17.0,NaN
118,2026-03-05 22:00:00,9999,9999.0,NaN
119,2026-03-05 23:00:00,9999,9999.0,NaN


In [15]:
om.dropna(subset=['SYM_H index'], inplace=True)

In [16]:
om

,SYM_H index,AE,DateTime
0,5.0,78.0,2026-01-01 00:00:00
1,6.0,104.0,2026-01-01 00:05:00
2,6.0,106.0,2026-01-01 00:10:00
3,4.0,105.0,2026-01-01 00:15:00
4,4.0,93.0,2026-01-01 00:20:00
...,...,...,...
16987,4.0,NaN,2026-02-28 23:35:00
16988,4.0,NaN,2026-02-28 23:40:00
16989,4.0,NaN,2026-02-28 23:45:00
16990,4.0,NaN,2026-02-28 23:50:00


In [121]:
[dt.strftime('%Y-%m-%d %X') for dt in w]

['2026-01-01 00:00:00', '2026-02-01 00:00:00', '2026-03-01 00:00:00']

In [91]:
pd.date_range(start='2021-02-12',end='2023-06-18', freq='MS')

DatetimeIndex(['2021-03-01', '2021-04-01', '2021-05-01', '2021-06-01',
               '2021-07-01', '2021-08-01', '2021-09-01', '2021-10-01',
               '2021-11-01', '2021-12-01', '2022-01-01', '2022-02-01',
               '2022-03-01', '2022-04-01', '2022-05-01', '2022-06-01',
               '2022-07-01', '2022-08-01', '2022-09-01', '2022-10-01',
               '2022-11-01', '2022-12-01', '2023-01-01', '2023-02-01',
               '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01'],
              dtype='datetime64[ns]', freq='MS')

In [87]:
today = datetime.date.today()
last_month_date = today - relativedelta(months=1)

In [ ]:
last_month_date.month

TypeError: 'int' object is not callable

In [85]:
load_prelim_feat()

2026-03-05
2026-02-05


In [81]:
x = stream_kyoto_dst('https://wdc.kugi.kyoto-u.ac.jp/dst_realtime/202602/dst2602.for.request')
x

,DateTime,DST
0,2026-02-01 00:00:00,-10
1,2026-02-01 01:00:00,-7
2,2026-02-01 02:00:00,-2
3,2026-02-01 03:00:00,0
4,2026-02-01 04:00:00,0
...,...,...
667,2026-02-28 19:00:00,10
668,2026-02-28 20:00:00,4
669,2026-02-28 21:00:00,6
670,2026-02-28 22:00:00,6


In [78]:
x

,DateTime,DST
0,2026-03-01 00:00:00,2
1,2026-03-01 01:00:00,2
2,2026-03-01 02:00:00,2
3,2026-03-01 03:00:00,2
4,2026-03-01 04:00:00,2
...,...,...
739,2026-03-31 19:00:00,9999
740,2026-03-31 20:00:00,9999
741,2026-03-31 21:00:00,9999
742,2026-03-31 22:00:00,9999


In [23]:
line = 'DST2603*01RRX020   0   3   0  -5  -6  -2  -3  -3   0  -2   0   1  -4  -5  -3   0   1   2   5   7   9   9  12  11  16   2'

id = (0,3)
year_yy = (3,5)
month = (5,7)
day = (8,10)
year_xx = (14,16)
dat = [(20,24),(24,28),(28,32),(32,36),(36,40),(40,44),(44,48),(48,52),
       (52,56),(56,60),(60,64),(64,68),(68,72),(72,76),(76,80),(80,84),(84,88),
       (88,92),(92,96),(96,100),(100,104),(104,108),(108,112),(112,116)]
avg = (116,120)

cols = [(0,3),(3,5),(5,7),(7,8),(8,10),(10,12),(12,13),(13,14),(14,16),
                 (16,20),(20,24),(24,28),(28,32),(32,36),(36,40),(40,44),(44,48),(48,52),
                 (52,56),(56,60),(60,64),(64,68),(68,72),(72,76),(76,80),(80,84),(84,88),
                 (88,92),(92,96),(96,100),(100,104),(104,108),(108,112),(112,116),
                 (116,120)]

In [32]:
data = np.array([float(line[slice(*s)]) for s in dat])

In [33]:
data

array([ 3.,  0., -5., -6., -2., -3., -3.,  0., -2.,  0.,  1., -4., -5.,
       -3.,  0.,  1.,  2.,  5.,  7.,  9.,  9., 12., 11., 16.])

In [ ]:
ww = pd.read_fwf(r'C:\Users\murph\Downloads\dst2603.for.request', colspecs=cols)

In [20]:
line[slice(*cols[0])]

'DST'

In [13]:
for col_val in cols:
    print(line[slice(*col_val)])

DST
26
03
*
01
RR
X
0
20
   0
   3
   0
  -5
  -6
  -2
  -3
  -3
   0
  -2
   0
   1
  -4
  -5
  -3
   0
   1
   2
   5
   7
   9
   9
  12
  11
  16
   2
